# CNNs: Clasificación de Imágenes

Las **Redes Neuronales Convolucionales (CNNs)** son la arquitectura fundamental para procesamiento de imágenes. Este notebook explora desde los conceptos básicos de convolución hasta el entrenamiento completo en CIFAR-10.

**Contenido:**
1. Qué es una convolución y cómo funciona
2. Arquitectura CNN completa
3. Dataset CIFAR-10 con data augmentation
4. Entrenamiento con learning rate scheduler
5. Evaluación detallada
6. Visualización de lo que aprende la CNN

**Requisitos:**
```bash
pip install torch torchvision matplotlib seaborn scikit-learn
```

## Qué es una convolución

Una **convolución** es una operación que desliza un filtro (kernel) sobre una imagen para detectar patrones locales.

```
Imagen de entrada (5x5)        Kernel (3x3)         Feature Map (3x3)
┌───┬───┬───┬───┬───┐         ┌───┬───┬───┐        ┌───┬───┬───┐
│ 1 │ 0 │ 1 │ 0 │ 1 │         │ 1 │ 0 │-1 │        │   │   │   │
├───┼───┼───┼───┼───┤   *     ├───┼───┼───┤   =    ├───┼───┼───┤
│ 0 │ 1 │ 0 │ 1 │ 0 │         │ 1 │ 0 │-1 │        │   │   │   │
├───┼───┼───┼───┼───┤         ├───┼───┼───┤        ├───┼───┼───┤
│ 1 │ 0 │ 1 │ 0 │ 1 │         │ 1 │ 0 │-1 │        │   │   │   │
├───┼───┼───┼───┼───┤         └───┴───┴───┘        └───┴───┴───┘
│ 0 │ 1 │ 0 │ 1 │ 0 │
├───┼───┼───┼───┼───┤
│ 1 │ 0 │ 1 │ 0 │ 1 │
└───┴───┴───┴───┴───┘
```

**Propiedades clave:**
- **Compartición de pesos**: el mismo filtro se aplica a toda la imagen (reduce parámetros)
- **Invarianza a traslación**: detecta el mismo patrón sin importar su posición
- **Jerarquía de features**: capas iniciales detectan bordes, capas profundas detectan objetos
- **Localidad**: cada neurona solo ve una región pequeña de la entrada (campo receptivo)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Create a simple grayscale image (8x8)
image = torch.zeros(1, 1, 8, 8)  # (batch, channels, height, width)
# Draw a vertical line
image[0, 0, 1:7, 3] = 1.0
image[0, 0, 1:7, 4] = 1.0

# Define different filters to detect different features
filters = {
    'Vertical edge': torch.tensor([[-1, 0, 1],
                                    [-1, 0, 1],
                                    [-1, 0, 1]], dtype=torch.float32),
    'Horizontal edge': torch.tensor([[-1, -1, -1],
                                      [ 0,  0,  0],
                                      [ 1,  1,  1]], dtype=torch.float32),
    'Sharpen': torch.tensor([[ 0, -1,  0],
                              [-1,  5, -1],
                              [ 0, -1,  0]], dtype=torch.float32),
    'Blur': torch.ones(3, 3, dtype=torch.float32) / 9.0
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

# Show original image
axes[0].imshow(image[0, 0].numpy(), cmap='gray')
axes[0].set_title('Imagen original')
axes[0].axis('off')

# Apply each filter and show result
for i, (name, kernel) in enumerate(filters.items()):
    # Reshape kernel for conv2d: (out_channels, in_channels, H, W)
    kernel_4d = kernel.unsqueeze(0).unsqueeze(0)
    
    # Apply convolution
    output = F.conv2d(image, kernel_4d, padding=1)
    
    axes[i+1].imshow(output[0, 0].detach().numpy(), cmap='RdBu_r')
    axes[i+1].set_title(name)
    axes[i+1].axis('off')

plt.suptitle('Efecto de diferentes filtros de convolución', fontsize=13)
plt.tight_layout()
plt.show()

# Show the kernels themselves
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, (name, kernel) in enumerate(filters.items()):
    axes[i].imshow(kernel.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
    axes[i].set_title(f'Kernel: {name}', fontsize=9)
    # Show values in each cell
    for row in range(3):
        for col in range(3):
            axes[i].text(col, row, f'{kernel[row, col]:.1f}',
                        ha='center', va='center', fontsize=8, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Kernels de convolución', fontsize=13)
plt.tight_layout()
plt.show()

## Arquitectura CNN: de píxeles a predicciones

Una CNN típica tiene la siguiente estructura:

```
Imagen → [Conv → BatchNorm → ReLU → Pool] × N → Flatten → [FC → ReLU] × M → Output
```

**Componentes:**
- **Conv2d**: extrae features locales con filtros aprendidos
- **BatchNorm**: normaliza las activaciones, estabiliza y acelera el entrenamiento
- **ReLU**: introduce no linealidad (`max(0, x)`)
- **MaxPool**: reduce la resolución espacial, añade invarianza a pequeñas traslaciones
- **Flatten**: convierte el tensor 3D (C, H, W) a 1D para las capas fully connected
- **FC (Linear)**: capas densas para clasificación final

A medida que avanzamos en la red, el tamaño espacial disminuye y el número de canales aumenta.

In [ ]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    """CNN for CIFAR-10 (32x32 RGB images, 10 classes)."""
    
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Feature extraction layers
        self.features = nn.Sequential(
            # Block 1: 3x32x32 -> 32x16x16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 2: 32x16x16 -> 64x8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 3: 64x8x8 -> 128x4x4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        
        # Classification layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create and inspect model
model = CNN(num_classes=10)
print("=== CNN Architecture ===")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Verify with dummy input
dummy_input = torch.randn(2, 3, 32, 32)  # Batch of 2 CIFAR images
output = model(dummy_input)
print(f"\nInput shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")

# Show output dimensions at each layer
print("\n=== Feature dimensions through the network ===")
x = dummy_input
for i, layer in enumerate(model.features):
    x = layer(x)
    if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
        print(f"After {layer.__class__.__name__:12s}: {list(x.shape)}")

## Dataset: CIFAR-10

**CIFAR-10** contiene 60,000 imágenes de 32x32 píxeles en 10 clases:
- airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

**Data Augmentation** es fundamental para mejorar la generalización:
- **RandomHorizontalFlip**: voltear horizontalmente (un avión sigue siendo un avión)
- **RandomCrop con padding**: recortar con ligero desplazamiento
- **ColorJitter**: variaciones de brillo, contraste y saturación
- **Normalize**: centrar los valores de píxeles en el rango correcto

> Solo aplicamos augmentation al conjunto de entrenamiento, nunca al de test.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

# CIFAR-10 normalization values
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

# Transforms for training (with augmentation)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Transforms for testing (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Download CIFAR-10
train_full = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

# Split training into train/val
train_size = int(0.9 * len(train_full))
val_size = len(train_full) - train_size
train_dataset, val_dataset = random_split(train_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

# Class names
classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Training samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples:       {len(test_dataset):,}")
print(f"Classes: {classes}")

# Visualize some training images
# Load without normalization for visualization
vis_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=transforms.ToTensor()
)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    img, label = vis_dataset[i]
    ax = axes[i // 8, i % 8]
    ax.imshow(img.permute(1, 2, 0).numpy())  # CHW -> HWC
    ax.set_title(classes[label], fontsize=9)
    ax.axis('off')

plt.suptitle('Ejemplos de CIFAR-10', fontsize=14)
plt.tight_layout()
plt.show()

## Entrenamiento

Para entrenar la CNN usamos:
- **CrossEntropyLoss**: la loss estándar para clasificación multi-clase
- **Adam optimizer**: combina las ventajas de AdaGrad y RMSProp
- **Learning rate scheduler**: reduce el learning rate cuando el validation loss se estanca
- **Early stopping manual**: guardamos el mejor modelo según validation accuracy

Monitoreamos tanto loss como accuracy en train y validation para detectar overfitting.

In [ ]:
import torch
import torch.nn as nn
import time

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Training on: {device}")

# Create model and move to device
model = CNN(num_classes=10).to(device)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

# Training loop
n_epochs = 25  # Increase for better results (50-100 epochs recommended)
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(n_epochs):
    start_time = time.time()
    
    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    # Calculate epoch metrics
    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn_cifar10.pt')
    
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1:2d}/{n_epochs} ({elapsed:.1f}s) | "
          f"Train: loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"Val: loss={val_loss:.4f} acc={val_acc:.4f}")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', linewidth=2)
ax1.plot(history['val_loss'], label='Validation', linewidth=2)
ax1.set_title('Loss durante el entrenamiento')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train', linewidth=2)
ax2.plot(history['val_acc'], label='Validation', linewidth=2)
ax2.set_title('Accuracy durante el entrenamiento')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluación

La evaluación final se hace sobre el test set (datos que el modelo nunca ha visto). Analizamos:

1. **Accuracy global**: porcentaje total de aciertos
2. **Accuracy por clase**: detectar clases problemáticas
3. **Confusion matrix**: ver qué clases se confunden entre sí
4. **Predicciones correctas e incorrectas**: inspección visual

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Load best model
model.load_state_dict(torch.load('best_cnn_cifar10.pt', weights_only=True))
model.eval()

# Evaluate on test set
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Overall accuracy
test_acc = (all_preds == all_labels).mean()
print(f"=== Test Accuracy: {test_acc:.4f} ===")

# Per-class accuracy
print(f"\n{'Class':<15} {'Accuracy':>10} {'Samples':>10}")
print("-" * 37)
for i, cls_name in enumerate(classes):
    mask = all_labels == i
    cls_acc = (all_preds[mask] == i).mean()
    print(f"{cls_name:<15} {cls_acc:>10.4f} {mask.sum():>10}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=classes)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Matriz de Confusión - CIFAR-10', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Visualize correct and incorrect predictions
vis_transform = transforms.ToTensor()
vis_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=False, transform=vis_transform
)

correct_idx = np.where(all_preds == all_labels)[0][:8]
incorrect_idx = np.where(all_preds != all_labels)[0][:8]

fig, axes = plt.subplots(2, 8, figsize=(16, 5))

for i, idx in enumerate(correct_idx):
    img, _ = vis_dataset[idx]
    axes[0, i].imshow(img.permute(1, 2, 0).numpy())
    axes[0, i].set_title(f'{classes[all_labels[idx]]}', fontsize=9, color='green')
    axes[0, i].axis('off')

for i, idx in enumerate(incorrect_idx):
    img, _ = vis_dataset[idx]
    axes[1, i].imshow(img.permute(1, 2, 0).numpy())
    axes[1, i].set_title(f'True: {classes[all_labels[idx]]}\nPred: {classes[all_preds[idx]]}',
                        fontsize=8, color='red')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Correctas', fontsize=11, color='green')
axes[1, 0].set_ylabel('Incorrectas', fontsize=11, color='red')
plt.suptitle('Predicciones del modelo', fontsize=14)
plt.tight_layout()
plt.show()

## Visualizar lo que aprende la CNN

Entender qué aprende una CNN es fundamental para diagnosticar problemas y mejorar el modelo.

**Técnicas de visualización:**
1. **Filtros de la primera capa**: se pueden interpretar directamente como detectores de bordes/colores
2. **Feature maps intermedios**: qué "ve" cada filtro al procesar una imagen concreta
3. **Grad-CAM** (más avanzado): qué regiones de la imagen son más importantes para la predicción

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

model.eval()
model_cpu = model.cpu()

# --- 1. Visualize first layer filters ---
# First conv layer weights: shape (32, 3, 3, 3)
first_conv = model_cpu.features[0]
filters = first_conv.weight.data.clone()

# Normalize filters for visualization
filters = (filters - filters.min()) / (filters.max() - filters.min())

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i in range(32):
    ax = axes[i // 8, i % 8]
    # Filter shape: (3, 3, 3) -> (3, 3, 3) -> permute to (3, 3, 3) HWC
    filt = filters[i].permute(1, 2, 0).numpy()
    ax.imshow(filt)
    ax.set_title(f'F{i}', fontsize=8)
    ax.axis('off')

plt.suptitle('Filtros de la primera capa convolucional (32 filtros de 3x3)', fontsize=13)
plt.tight_layout()
plt.show()

# --- 2. Visualize feature maps of intermediate layers ---
# Get a sample image
sample_img, sample_label = test_dataset[0]
sample_batch = sample_img.unsqueeze(0)  # Add batch dimension

# Hook to capture intermediate outputs
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks on conv layers
hooks = []
conv_names = []
for name, layer in model_cpu.features.named_modules():
    if isinstance(layer, torch.nn.Conv2d):
        hooks.append(layer.register_forward_hook(hook_fn(f'conv_{name}')))
        conv_names.append(f'conv_{name}')

# Forward pass
with torch.no_grad():
    _ = model_cpu(sample_batch)

# Remove hooks
for h in hooks:
    h.remove()

# Plot feature maps for first 3 conv layers
layers_to_show = conv_names[:3]
fig, axes = plt.subplots(len(layers_to_show), 8, figsize=(14, 5))

for row, layer_name in enumerate(layers_to_show):
    feat_map = activations[layer_name][0]  # Remove batch dim
    for col in range(8):
        ax = axes[row, col] if len(layers_to_show) > 1 else axes[col]
        ax.imshow(feat_map[col].numpy(), cmap='viridis')
        if col == 0:
            ax.set_ylabel(layer_name, fontsize=9, rotation=0, labelpad=60)
        ax.axis('off')

plt.suptitle(f'Feature maps para "{classes[sample_label]}"', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Image class: {classes[sample_label]}")
for name in conv_names:
    shape = activations[name].shape
    print(f"{name}: {list(shape)} ({shape[1]} filters, {shape[2]}x{shape[3]} spatial)")

## Siguientes pasos: Transfer Learning

Entrenar una CNN desde cero requiere muchos datos y tiempo de GPU. En la práctica, casi siempre es mejor usar **Transfer Learning**:

1. **Feature Extraction**: usar un modelo preentrenado (ResNet, EfficientNet) como extractor de features, solo entrenar la última capa
2. **Fine-Tuning**: descongelar algunas capas del modelo preentrenado y reentrenarlas con un learning rate bajo

**Ventajas:**
- Funciona con pocos datos (incluso cientos de imágenes)
- Convergencia mucho más rápida
- Mejor generalización gracias a las features aprendidas de ImageNet (1.2M de imágenes)

El siguiente notebook cubre Transfer Learning en detalle.